In [197]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
rng = np.random.default_rng(67)

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

X_train, X_test, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=67,
    stratify=y           # photos of a person will be in both sets
)

In [198]:
y_one = np.where(y_train == 1, 1, -1)
y_val = np.where(y_val == 1, 1, -1)

In [201]:
def initiateWeights(y):
    sumOnes = np.sum(y==True)
    others = len(y) - sumOnes
    weights = np.where(y==True, 1/sumOnes, 1/others)
    return weights

In [202]:
def normalizeWeights(weights):
    norm_weights = weights/np.sum(weights)
    return norm_weights

In [203]:
weights = initiateWeights(y_one)
weights1 = normalizeWeights(weights)
weights1.sum()

np.float64(1.0)

In [204]:
weak_learner = DecisionTreeClassifier(max_depth=1)

In [ ]:
def calculateThreshold(alpha):
    return 0.5*(np.sum(alpha))


In [ ]:
x = 0
amountOfSay = []
stumps_list = []
f = 0
d = 0
iteration = 0
weights = initiateWeights(y_one)
while(f>0.1 or d<0.9): 
    iteration += 1
    weights = normalizeWeights(weights)
    
    new_learner = weak_learner.fit(X_train, y_one, sample_weight=weights)
    
    stumps_list.append(new_learner)
    
    predictions = stumps_list[-1].predict(X_train)
    
    misclassified = (y_one !=predictions)
    
    epsilon = np.sum(weights[misclassified])/np.sum(weights)
    
    #amount of say
    alpha = 0.5*np.log((1-epsilon)/(epsilon + 1e-10))
    
    #weight actualization
    weights = weights*np.exp(-alpha * y_one * predictions) 
    amountOfSay.append(alpha)
    threshold = calculateThreshold(amountOfSay)
    pred = np.zeros(len(X_test))
    for a, stump in zip(amountOfSay, stumps_list):
        pred += a*stump.predict(X_test)

    final_results = np.where(pred>=threshold, 1, -1)
    
    fp = np.sum((y_val== -1) & (final_results == 1))
    tp = np.sum((y_val == 1) & (final_results == 1))
    f = fp/sum(y_val == -1)
    d = tp/sum(y_val==1)
    if(not iteration%10):
        print(iteration)
        print(f)
        print(d) 
    x = x+1
print(f)
print(d)    




[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1  1]
[-1]
[-1  1]
[-1  1]


KeyboardInterrupt: 